# Kaggle → Google Drive: tạo train/val/test

Notebook này tải **toàn bộ** một phiên bản dataset Kaggle về ổ tạm `/content`, kiểm tra ảnh, loại file trùng SHA-256, gom ảnh theo nhóm rồi chia đủ `train/val/test`. Sau đó notebook tạo ZIP, manifest, báo cáo nguồn và checksum rồi lưu vào Google Drive.

> Phạm vi: đây là `v0 Kaggle smoke dataset` để hoàn chỉnh và kiểm thử pipeline model/software. Vì cả ba split cùng miền Kaggle, kết quả chưa được dùng làm KPI cuối cho ảnh camera thực tế.

## Tạo và thêm Kaggle API token vào Colab

1. Mở [Kaggle Settings](https://www.kaggle.com/settings), tìm mục **API** và chọn **Generate New Token**.
2. Trong Colab, bấm biểu tượng **chìa khóa / Secrets** ở thanh bên trái.
3. Nếu Kaggle cấp access token (dạng `KGAT_...`): tạo secret tên chính xác `KAGGLE_API_TOKEN`, dán token làm Value.
4. Nếu Kaggle chỉ tải xuống tệp `kaggle.json`: mở tệp, tạo thêm hai secret `KAGGLE_USERNAME` (username) và `KAGGLE_KEY` (key).
5. Bật **Notebook access** cho tất cả secret vừa tạo.
6. Không dán token vào ô code, không lưu token vào Drive và không commit token lên Git.

Dataset công khai đôi khi tải được không cần đăng nhập, nhưng notebook vẫn yêu cầu secret để quy trình ổn định và dùng được với dataset cần chấp nhận điều khoản.

In [ ]:
REPO_URL = "https://github.com/DangHung1110/IoT-CK.git"
GIT_REF = "feat/model-training-pipeline"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/parcel_sorter"  # @param {type:"string"}
KAGGLE_DATASET_HANDLE = "owner/dataset/versions/1"  # @param {type:"string"}
KAGGLE_SOURCE_URL = "https://www.kaggle.com/datasets/owner/dataset"  # @param {type:"string"}
KAGGLE_LICENSE = "REPLACE_WITH_LICENSE_FROM_KAGGLE_PAGE"  # @param {type:"string"}
DATASET_VERSION = "v1"  # @param {type:"string"}
CLASS_MAP = '{"ok":"ok","rach":"rach","mop":"mop","vo":"vo","uot":"uot"}'  # @param {type:"string"}
GROUP_REGEX = ""  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}
TRAIN_RATIO = 0.70  # @param {type:"number"}
VAL_RATIO = 0.15  # @param {type:"number"}
TEST_RATIO = 0.15  # @param {type:"number"}

### Trước khi chạy tiếp

- Thay `KAGGLE_DATASET_HANDLE` bằng handle có khóa version, ví dụ `owner/dataset-name/versions/3`.
- Chép đúng License từ trang Kaggle vào `KAGGLE_LICENSE`.
- `CLASS_MAP` ánh xạ **tên thư mục gốc trong Kaggle** sang 5 nhãn chuẩn: `ok`, `rach`, `mop`, `vo`, `uot`. Có thể có nhiều thư mục nguồn cùng trỏ vào một nhãn.
- Nếu tên file chứa ID của cùng kiện hàng/video/chuỗi ảnh, điền regex có ít nhất một capture group vào `GROUP_REGEX`, ví dụ `parcel_(\d+)`. Nếu để trống, pipeline nhóm theo average-hash của ảnh.
- Mỗi lớp cần tối thiểu 3 nhóm độc lập để cả train, val và test đều có dữ liệu.

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    kaggle_token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    kaggle_token = None
if kaggle_token:
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
else:
    try:
        kaggle_username = userdata.get("KAGGLE_USERNAME")
        kaggle_key = userdata.get("KAGGLE_KEY")
    except Exception:
        kaggle_username = kaggle_key = None
    if not kaggle_username or not kaggle_key:
        raise RuntimeError(
            "Thiếu Colab Secret: dùng KAGGLE_API_TOKEN hoặc cặp KAGGLE_USERNAME/KAGGLE_KEY"
        )
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key
    del kaggle_username, kaggle_key
del kaggle_token
print("Google Drive đã mount; thông tin Kaggle đã nạp an toàn từ Colab Secrets.")

In [ ]:
import shutil
import subprocess
from pathlib import Path

repo_dir = Path("/content/IoT-CK")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo_dir)],
    check=True,
)
%pip install -q kagglehub==1.0.3 "pydantic>=2.10,<3" "pillow>=13,<14"
%pip install -q -e {repo_dir} --no-deps

In [ ]:
import kagglehub

download_root = Path("/content/kaggle_download")
if download_root.exists():
    shutil.rmtree(download_root)
download_root.mkdir()
downloaded_path = Path(
    kagglehub.dataset_download(
        KAGGLE_DATASET_HANDLE, output_dir=str(download_root), force_download=True
    )
)
if not downloaded_path.is_dir():
    raise RuntimeError(f"Kaggle không trả về thư mục dataset: {downloaded_path}")
print(f"Dataset đã tải vào ổ tạm Colab: {downloaded_path}")

In [ ]:
import json

from parcel_sorter.data.kaggle_import import (
    KaggleImportConfig,
    SplitRatios,
    prepare_kaggle_smoke_dataset,
)

class_folder_map = json.loads(CLASS_MAP)
config = KaggleImportConfig(
    dataset_handle=KAGGLE_DATASET_HANDLE,
    source_url=KAGGLE_SOURCE_URL,
    license_name=KAGGLE_LICENSE,
    dataset_version=DATASET_VERSION,
    class_folder_map=class_folder_map,
    split_seed=SPLIT_SEED,
    ratios=SplitRatios(train=TRAIN_RATIO, val=VAL_RATIO, test=TEST_RATIO),
    group_regex=GROUP_REGEX or None,
)
workspace = Path(f"/content/parcel_sorter_data_{DATASET_VERSION}")
if workspace.exists():
    shutil.rmtree(workspace)
prepared = prepare_kaggle_smoke_dataset(
    source_root=downloaded_path, workspace=workspace, config=config
)
print(json.dumps(prepared.report.class_counts_by_split, indent=2))
print(f"Accepted: {prepared.report.accepted_count}")
print(f"Duplicates removed: {prepared.report.duplicate_count}")
print(f"Invalid: {prepared.report.invalid_count}; unmapped: {prepared.report.unmapped_count}")

## Cổng kiểm tra trước khi ghi Drive

Xem bảng số lượng vừa in: cả 5 lớp phải có số ảnh lớn hơn 0 trong cả `train`, `val`, `test`. Nếu mapping sai hoặc dữ liệu thiếu nhóm, cell chuẩn bị sẽ dừng và **chưa ghi gì vào Drive**. Chỉ chạy cell cuối khi kết quả hợp lý.

In [ ]:
from parcel_sorter.data.kaggle_import import persist_kaggle_smoke_dataset

persisted = persist_kaggle_smoke_dataset(prepared=prepared, drive_root=Path(DRIVE_ROOT))
print(f"ARCHIVE_PATH={persisted.dataset_archive}")
print(f"MANIFEST_PATH={persisted.manifest}")
print(f"DATASET_ARCHIVE_SHA256={prepared.report.dataset_archive_sha256}")
print(f"DATASET_MANIFEST_SHA256={prepared.report.manifest_sha256}")
print(f"IMPORT_REPORT={persisted.report}")
print("Hoàn tất. Copy hai SHA-256 trên sang notebook train.")